In [4]:
import pandas as pd
from pathlib import Path

project = Path.cwd().parent

features = pd.read_csv(
    project / "data" / "processed" / "customer_features.csv"
)

features["SIGNUP_DATE"] = pd.to_datetime(features["SIGNUP_DATE"])
features = features.sort_values("SIGNUP_DATE").reset_index(drop=True)

cutoff_date = features["SIGNUP_DATE"].quantile(0.8)

train_mask = features["SIGNUP_DATE"] <= cutoff_date
test_mask = features["SIGNUP_DATE"] > cutoff_date

target = "CHURNED"

X = features.drop(columns=[target, "CUSTOMER_ID", "SIGNUP_DATE"])
y = features[target].astype(int)

categorical_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_columns = X.select_dtypes(exclude=["object", "string"]).columns.tolist()

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

train_customers = features.loc[train_mask, "CUSTOMER_ID"].values
test_customers = features.loc[test_mask, "CUSTOMER_ID"].values

validation_summary = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "Customers": [len(X_train), len(X_test)],
    "Churned": [y_train.sum(), y_test.sum()],
    "Churn Rate": [
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

validation_summary

,Dataset,Customers,Churned,Churn Rate
0,Train,1780,615,34.55
1,Validation,445,47,10.56
